<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_02_04_hpc_parallel_processing_doParallel_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Data Processing and Analysis with {doParallel}


The {doParallel} package in R is a powerful tool for parallel computing, enabling users to execute tasks concurrently to improve computational efficiency, especially for time-consuming operations. It serves as a backend for the {foreach} package, providing a mechanism to distribute computations across multiple CPU cores or processors, thereby reducing execution time for tasks that can be parallelized.

This package is part of the R ecosystem for parallel processing. It integrates the {foreach} package, which provides a looping construct for executing iterative tasks, with the {parallel} package, a base R package introduced in R 2.14.0 for parallel computation. {doParallel} simplifies the process of running loops in parallel by leveraging multiple CPU cores, making it particularly useful for tasks like simulations, large-scale data processing, or computationally intensive statistical analyses.

## Key features of {doParallel}:

- **Parallel Execution**: Distributes loop iterations across multiple cores or processors.
- **Ease of Use**: Works seamlessly with {foreach}, allowing users to write parallel loops with minimal code changes compared to standard sequential loops.
- **Cross-Platform Support**: Supports both Windows and Unix-like systems (e.g., Linux, macOS) through different parallelization mechanisms.
- **Scalability**: Can utilize multiple cores on a single machine, making it accessible for users without access to a computing cluster.


## How {doParallel} Works

{doParallel} combines the {foreach} package's looping framework with the {parallel} package's ability to manage parallel workers. It sets up a parallel backend that allows {foreach} loops to execute iterations concurrently across multiple CPU cores. Below is a step-by-step explanation of how it works:


## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab's Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there. Adjust `dataFolder` in the data-setup cell to point at your Parquet file (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Check and Install Required R Packages

The cells below mirror the Quarto tutorial: define packages, install to Drive (Colab), verify, load.


### Define required packages


In [ ]:
%%R
packages <- c(
  'tidyverse',
  'data.table',
  'feather',
  'arrow',
  'doParallel',
  'foreach'
)


### Install missing packages (Google Drive library)


In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")


### Verify installation


In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load packages


In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages


### Check loaded packages


In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


### Set Up the Parallel Backend

{doParallel} uses the {parallel} package to create a cluster of worker processes. You specify the number of CPU cores to use, and {doParallel} registers these as a parallel backend for {foreach}.

- On Unix-like systems, {doParallel} can use forking (via multicore workers) depending on configuration.
- On Windows, it often uses PSOCK (Parallel Socket Cluster) to spawn separate R processes for parallel execution.

**Note (Colab / `rpy2`):** If parallel backend registration fails in a hosted environment, try reducing the number of cores or run the notebook locally.


In [ ]:
%%R
library(parallel)
library(doParallel)
library(foreach)
# Detect the number of available CPU cores
cores <- max(1L, detectCores() - 1L)  # Leave one core free to avoid overloading
# Register the parallel backend
registerDoParallel(cores)
cat("Registered workers:", cores, "\n")


### Write a {foreach} Loop

The {foreach} package provides a looping construct similar to a `for` loop but designed for parallel execution. You write the loop using `%dopar%` to indicate that iterations should run in parallel.

- `.combine = c`: Specifies how to combine results from parallel iterations (here, results are concatenated into a vector).
- `%dopar%`: Instructs {foreach} to execute the loop in parallel using the registered {doParallel} backend.


In [ ]:
%%R
# Example: Parallel computation of squares
result <- foreach(i = 1:10, .combine = c) %dopar% {
  i^2  # Each iteration computes the square of i
}
print(result)


### Execute and Collect Results

Each iteration of the {foreach} loop is distributed to a worker process (or core). The results are collected and combined as specified (e.g., into a vector, list, or matrix). In the example above, `result` contains the squares of numbers 1 to 10.


### Clean Up

After completing the parallel tasks, it is good practice to stop the implicit cluster to free up system resources:


In [ ]:
%%R
stopImplicitCluster()


### Example: Parallel Matrix Multiplication

Practical example demonstrating {doParallel} for matrix multiplication across multiple cores. **Warning:** `n <- 1000` builds two 1000×1000 matrices and is memory-intensive; reduce `n` (e.g., 200) on machines with limited RAM.


In [ ]:
%%R
library(parallel)
library(doParallel)
library(foreach)
# Set up parallel backend
cores <- max(1L, detectCores() - 1L)
registerDoParallel(cores)
# Create sample matrices (reduce n on low-memory environments)
n <- 1000
mat1 <- matrix(rnorm(n * n), nrow = n)
mat2 <- matrix(rnorm(n * n), nrow = n)
# Parallel matrix multiplication
result <- foreach(i = 1:nrow(mat1), .combine = rbind) %dopar% {
  row <- mat1[i, ]
  apply(mat2, 1, function(col) sum(row * col))
}
stopImplicitCluster()
# View result
print(dim(result))  # Should be n x n


### Key Components and Options

- **Parallel Backend**: {doParallel} relies on the {parallel} package's worker management to run iterations concurrently.
- **.combine Argument**: Controls how results are aggregated (e.g., `c` for vectors, `rbind` for matrices, `list` for lists).
- **.packages Argument**: Specifies additional packages to load in each worker process if needed (e.g., `.packages = "dplyr"`).
- **Error Handling**: Use `.errorhandling = "remove"` or `"stop"` to handle errors in parallel iterations gracefully.


### How It Improves Performance

By distributing iterations across multiple cores, {doParallel} reduces execution time for tasks that are embarrassingly parallel (i.e., iterations are independent). Overhead from worker setup and data transfer may reduce gains for very small tasks.


### Limitations

- **Overhead**: Creating and managing worker processes introduces some overhead, so parallelization is only beneficial for computationally intensive tasks.
- **Memory Usage**: Each worker process may duplicate data, increasing memory demands.
- **Windows Constraints**: Windows often uses PSOCK clusters, which can be slower than forking on Unix-like systems due to process creation overhead.
- **Dependencies**: Tasks must be independent; {doParallel} does not handle inter-iteration dependencies well.


## Parallel Data Processing and Analysis with {doParallel}

This section demonstrates how to use the {doParallel} package in R for parallel computing applied to data processing and basic statistical analyses on a large dataset. We use the NYC Yellow Taxi Trip Records for January 2023, stored in the Parquet file `yellow_tripdata_2023-01.parquet`. This dataset contains approximately 3 million rows of taxi trip data, making it suitable for showcasing parallel processing to speed up computations.


### Data

#### Set `dataFolder` for Colab (edit path)


In [ ]:
%%R
# Colab: set folder containing yellow_tripdata_2023-01.parquet
dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
dataFolder <- sub("/$", "", dataFolder)
dataFolder <- paste0(dataFolder, "/")


#### Local / repo path (edit if running outside this project)


In [ ]:
%%R
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
df <- read_parquet(paste0(dataFolder, "yellow_tripdata_2023-01.parquet"))
dim(df)


### Setting Up Parallel Backend

Detect available cores and register the parallel backend. We use all but one core to avoid system overload.


In [ ]:
%%R
cores <- max(1L, parallel::detectCores() - 1L)
registerDoParallel(cores)


### Data Processing with {doParallel}

For large datasets, parallel processing can accelerate data cleaning and feature engineering. Here, we add a new column for trip duration (in minutes) and filter out invalid rows. We split the data into chunks, process each chunk in parallel, and combine the results.

First, split the data into a list of chunks (e.g., 4 chunks for simplicity).


In [ ]:
%%R
num_chunks <- 4
chunk_size <- floor(nrow(df) / num_chunks)  # Use floor to ensure integer division
chunks <- lapply(1:num_chunks, function(i) {
  start <- as.integer((i - 1) * chunk_size + 1)
  end <- if (i == num_chunks) nrow(df) else as.integer(i * chunk_size)
  df[start:end, ]
})


Now, use {foreach} with `%dopar%` to process each chunk in parallel: calculate trip duration, filter invalid data (`trip_distance > 0`, `fare_amount > 0`, `passenger_count > 0`).


In [ ]:
%%R
processed_chunks <- foreach(chunk = chunks, .combine = rbind, .packages = "dplyr") %dopar% {
  chunk %>%
    mutate(trip_duration = as.numeric(difftime(tpep_dropoff_datetime, tpep_pickup_datetime, units = "mins"))) %>%
    filter(trip_distance > 0, fare_amount > 0, passenger_count > 0, trip_duration > 0)
}

dim(processed_chunks)  # Check dimensions after processing
head(processed_chunks)


### Descriptive Statistics

Descriptive statistics like mean, median, min, max, and standard deviation can be computed on subsets or via bootstrapping in parallel for efficiency and to estimate variability.

Example: Compute bootstrap means for key variables (`trip_distance`, `fare_amount`, `tip_amount`) in parallel. We perform 100 bootstrap resamples per variable.


In [ ]:
%%R
variables <- c("trip_distance", "fare_amount", "tip_amount")
boot_results <- foreach(var = variables, .combine = rbind) %dopar% {
  boot_means <- replicate(100, mean(sample(processed_chunks[[var]], replace = TRUE), na.rm = TRUE))
  data.frame(variable = var, mean = mean(boot_means), sd = sd(boot_means))
}

boot_results


For a more granular approach, compute summary statistics on grouped data (e.g., by `payment_type`) in parallel.


In [ ]:
%%R
payment_types <- unique(processed_chunks$payment_type)
group_stats <- foreach(pt = payment_types, .combine = rbind, .packages = "dplyr") %dopar% {
  subset <- filter(processed_chunks, payment_type == pt)
  data.frame(
    payment_type = pt,
    mean_distance = mean(subset$trip_distance, na.rm = TRUE),
    median_fare = median(subset$fare_amount, na.rm = TRUE),
    sd_duration = sd(subset$trip_duration, na.rm = TRUE)
  )
}

group_stats


### Correlation Analysis

For correlations between multiple pairs of variables, parallelize the computation across pairs.

Example: Compute Pearson correlations between selected numeric variables in parallel.


In [ ]:
%%R
num_vars <- c("trip_distance", "fare_amount", "tip_amount", "total_amount", "trip_duration")
cor_pairs <- combn(num_vars, 2)  # All unique pairs

cor_results <- foreach(i = 1:ncol(cor_pairs), .combine = rbind) %dopar% {
  var1 <- cor_pairs[1, i]
  var2 <- cor_pairs[2, i]
  corr <- cor(processed_chunks[[var1]], processed_chunks[[var2]], use = "complete.obs")
  data.frame(var1 = var1, var2 = var2, correlation = corr)
}

cor_results


### Simple Linear Regression

Simple regression (`fare_amount ~ trip_distance`) is quick; here we run bootstrapped regressions in parallel to estimate coefficient variability.


In [ ]:
%%R
boot_reg <- foreach(i = 1:100, .combine = rbind) %dopar% {
  sample_idx <- sample(1:nrow(processed_chunks), replace = TRUE)
  model <- lm(fare_amount ~ trip_distance, data = processed_chunks[sample_idx, ])
  coef(model)
}

# Summarize bootstrap results
simple_reg_summary <- data.frame(
  intercept_mean = mean(boot_reg[, 1]), intercept_sd = sd(boot_reg[, 1]),
  slope_mean = mean(boot_reg[, 2]), slope_sd = sd(boot_reg[, 2])
)
simple_reg_summary


### Multiple Linear Regression

Similarly, for multiple regression (`fare_amount ~ trip_distance + passenger_count + trip_duration`), use parallel bootstrapping.


In [ ]:
%%R
boot_multi_reg <- foreach(i = 1:100, .combine = rbind) %dopar% {
  sample_idx <- sample(1:nrow(processed_chunks), replace = TRUE)
  model <- lm(fare_amount ~ trip_distance + passenger_count + trip_duration, data = processed_chunks[sample_idx, ])
  coef(model)
}

# Summarize
multi_reg_summary <- data.frame(
  intercept_mean = mean(boot_multi_reg[, 1]), intercept_sd = sd(boot_multi_reg[, 1]),
  distance_slope_mean = mean(boot_multi_reg[, 2]), distance_slope_sd = sd(boot_multi_reg[, 2]),
  passenger_slope_mean = mean(boot_multi_reg[, 3]), passenger_slope_sd = sd(boot_multi_reg[, 3]),
  duration_slope_mean = mean(boot_multi_reg[, 4]), duration_slope_sd = sd(boot_multi_reg[, 4])
)
multi_reg_summary


### Clean Up

Stop the parallel backend when finished.


In [ ]:
%%R
stopImplicitCluster()


## Summary and Conclusion

The {doParallel} package is an accessible and efficient way to implement parallel computing in R, particularly for iterative tasks that can be split across multiple cores. By registering a parallel backend and using {foreach} with `%dopar%`, users can significantly speed up computations for data processing, statistical analyses, and simulations. This tutorial demonstrated how to leverage {doParallel} for various data processing and analysis tasks on a large dataset. At the end of the analysis, always remember to stop the parallel cluster to free up system resources.


## References

1. **doParallel CRAN Documentation** — Weston, S., & Revolution Analytics. (2022). *doParallel: Foreach Parallel Adaptor*. [CRAN: doParallel](https://cran.r-project.org/package=doParallel)

2. **Getting Started with doParallel** — Weston, S. (2019). *doParallel Vignette*. [gettingstartedParallel.pdf](https://cran.r-project.org/web/packages/doParallel/vignettes/gettingstartedParallel.pdf)

3. **Parallel R Book** — McCallum, Q. E., & Weston, S. (2011). *Parallel R*. O'Reilly Media.

4. **Parallel Computing in R** — Schmidberger, M., et al. (2009). *Journal of Statistical Software*, 31(1). [doi:10.18637/jss.v031.i01](https://doi.org/10.18637/jss.v031.i01)
